# ML 03 — Feature engineering and leakage audit

This notebook builds the feature table from the freshly processed data, audits which columns are allowed as predictors, checks redundancy, and writes a clean modeling table. It does not use any file from `results/` as input.

In [ ]:
from pathlib import Path
import os

# Works whether Jupyter starts in the repository root or this notebook's folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Open this notebook from inside the T006 project folder.")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

processed_path = DATA_PROCESSED / "layout_level_dataset.csv"
if not processed_path.exists():
    raise FileNotFoundError("Run ML_01_dataset_understanding_data_dictionary.ipynb first; it builds this processed dataset from raw data.")
df = pd.read_csv(processed_path)
TARGET = "target_U_canopy_mean_m_s"
NUMERIC_FEATURES = ["plan_area_density", "plan_area_density_sq", "height_std_m", "height_max_m", "height_min_m", "height_range_m", "height_skew", "frac_taller_than_mean", "wind_direction_deg"]
CATEGORICAL_FEATURES = ["configuration", "horizontal_configuration", "density_class"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
BASELINE_NUMERIC = ["plan_area_density", "height_mean_m", "height_std_m", "height_max_m", "wind_direction_deg", "building_count"]
BASELINE_FEATURES = BASELINE_NUMERIC + CATEGORICAL_FEATURES
print("Dataset shape:", df.shape)
print("Target:", TARGET)

## 1. Validate engineered features

In [ ]:
# Recompute the density square in this notebook so feature engineering is executable and auditable.
engineered = df.copy()
engineered["plan_area_density_sq"] = engineered["plan_area_density"] ** 2

# Direction is circular. Keep the original direction column for continuity with the current baseline,
# and also calculate sine/cosine encodings for analysis. The baseline model below uses the listed feature whitelist.
angle_rad = np.deg2rad(engineered["wind_direction_deg"])
engineered["wind_direction_sin"] = np.sin(angle_rad)
engineered["wind_direction_cos"] = np.cos(angle_rad)
engineered["height_coefficient_variation"] = np.where(
    engineered["height_mean_m"] != 0,
    engineered["height_std_m"] / engineered["height_mean_m"],
    0.0,
)

assert np.allclose(engineered["plan_area_density_sq"], engineered["plan_area_density"] ** 2)
assert engineered["simulation_id"].is_unique
assert engineered[TARGET].notna().all()
print("Feature engineering checks passed.")
display(engineered[["simulation_id", "plan_area_density", "plan_area_density_sq", "wind_direction_deg", "wind_direction_sin", "wind_direction_cos", "height_coefficient_variation"]].head())

## 2. Predictor whitelist and leakage checks

In [ ]:
allowed_predictors = FEATURES
forbidden_predictors = {TARGET, "simulation_id", "layout_id", "family_id", "configuration_metadata_label", "height_mean_m", "building_count"}
assert TARGET not in allowed_predictors
assert not (set(allowed_predictors) & forbidden_predictors)
missing_columns = sorted(set(allowed_predictors) - set(engineered.columns))
assert not missing_columns, f"Missing model features: {missing_columns}"

print("Allowed predictors:")
for feature in allowed_predictors:
    print(" -", feature)
print()
print("Excluded from predictors:")
for feature in sorted(forbidden_predictors):
    print(" -", feature)
print()
print("Reminder: family_id and layout_id are used only to group validation folds, never as model inputs.")

## 3. Correlation and categorical feature audit

In [ ]:
corr_cols = NUMERIC_FEATURES + [TARGET]
corr = engineered[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=75, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title("Numeric feature correlation matrix")
fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()
print("Categorical levels:")
for col in CATEGORICAL_FEATURES:
    print(col, sorted(engineered[col].dropna().unique().tolist()))

## 4. Save the modeling table

In [ ]:
modeling_path = DATA_PROCESSED / "feature_engineered_dataset.csv"
engineered.to_csv(modeling_path, index=False)
print("Created:", modeling_path)
print("Rows:", len(engineered), "| columns:", len(engineered.columns))
print("The model feature whitelist remains explicit in this notebook; diagnostic columns added above are not silently added to the baseline models.")